# 📈 Real-Time Portfolio Analytics Dashboard

A Python dashboard that tracks a stock portfolio against live market data: cost basis, profit and loss, allocation, benchmark performance and risk metrics, with an auto-refreshing live mode.

**This notebook runs on a sample portfolio** for demonstration. To track your own, edit the `holdings` table in cell 1.

**How to run:** Runtime > Run all. Cell 4 (live mode) keeps running until you click its stop button.

| Cell | What it does |
|---|---|
| 1 | Loads holdings, pulls price history and current prices, builds the holdings table |
| 2 | Snapshot chart: average cost vs. current price |
| 3 | Performance analytics: growth vs. S&P 500, Sharpe ratio, drawdown, beta, XIRR |
| 4 | Live mode: refreshes prices, charts and table on a timer, aware of market hours |

*For demonstration only. Not financial advice.*

## 1. Setup, holdings and data pipeline

In [ ]:
import time
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import clear_output, display

# ✏️ Sample portfolio (edit to track your own)
# buy_date: when the position was opened. avg_cost: leave as None to use that day's closing price.
# Enter shares in today's split-adjusted terms.
holdings = pd.DataFrame({
    "ticker":   ["VOO",        "AAPL",       "MSFT",       "NVDA",       "KO",         "JPM",        "AMD",        "COST"],
    "shares":   [2,            3,            2,            10,           8,            3,            4,            1],
    "buy_date": ["2024-01-08", "2024-03-04", "2024-06-03", "2024-09-03", "2024-11-04", "2025-02-03", "2025-04-01", "2025-07-01"],
    "avg_cost": [None] * 8,
})

BENCHMARK = "SPY"        # compare against the S&P 500
RISK_FREE_RATE = 0.04    # annual, used for the Sharpe ratio
TZ = "America/Los_Angeles"


def money(v):
    """Format dollars with the sign in front: +$1.23 / -$1.23."""
    return f"{'-' if v < 0 else '+'}${abs(v):,.2f}"


def get_prices(tickers, intraday=False, extended=False):
    """Latest price per ticker. Daily close by default, 1-minute bars when intraday=True.
    Any ticker that comes back empty is retried one at a time."""
    prices = pd.Series(np.nan, index=tickers, dtype=float)
    try:
        if intraday:
            data = yf.download(tickers, period="1d", interval="1m", prepost=extended,
                               auto_adjust=False, progress=False)
        else:
            data = yf.download(tickers, period="5d", auto_adjust=False, progress=False)
        close = data["Close"]
        if isinstance(close, pd.Series):
            close = close.to_frame(tickers[0])
        prices.update(close.ffill().iloc[-1])
    except Exception as e:
        print(f"Bulk download failed ({e}), trying one at a time")
    for t in prices.index[prices.isna()]:
        try:
            prices[t] = float(yf.Ticker(t).fast_info["last_price"])
        except Exception:
            pass
    return prices


def revalue(df, prices):
    """Recalculate value, gain and return. Keeps the previous price if a lookup failed."""
    df = df.copy()
    new = df["ticker"].map(prices)
    df["price"] = new.fillna(df["price"]) if "price" in df else new
    df["cost_basis"] = df["shares"] * df["avg_cost"]
    df["value"] = df["shares"] * df["price"]
    df["gain"] = df["value"] - df["cost_basis"]
    df["return_pct"] = df["gain"] / df["cost_basis"]
    return df


def summary(df):
    spent, value, gain = df["cost_basis"].sum(), df["value"].sum(), df["gain"].sum()
    return f"Spent ${spent:,.2f}   |   Value ${value:,.2f}   |   P/L {money(gain)} ({gain / spent:+.2%})"


def style_table(df):
    cols = {"ticker": "Ticker", "shares": "Shares", "avg_cost": "Avg cost", "price": "Price",
            "cost_basis": "Spent", "value": "Value", "gain": "Gain/loss", "return_pct": "Return"}
    t = df.sort_values("value", ascending=False)[list(cols)].rename(columns=cols).set_index("Ticker")
    color = lambda v: f"color: {'#16803C' if v >= 0 else '#B42318'}; font-weight: 600"
    styler = t.style.format({"Shares": "{:,.4f}", "Avg cost": "${:,.2f}", "Price": "${:,.2f}",
                             "Spent": "${:,.2f}", "Value": "${:,.2f}", "Gain/loss": money,
                             "Return": "{:+.2%}"})
    apply_map = styler.map if hasattr(styler, "map") else styler.applymap   # works on old and new pandas
    return apply_map(color, subset=["Gain/loss", "Return"])


# ---------- Data pipeline ----------
holdings["buy_date"] = pd.to_datetime(holdings["buy_date"])
tickers = holdings["ticker"].tolist()

# Daily price history since the first purchase (split-adjusted closes)
raw = yf.download(tickers + [BENCHMARK], start=holdings["buy_date"].min() - pd.Timedelta(days=7),
                  auto_adjust=False, progress=False)
history = raw["Close"].ffill()
history.index = pd.to_datetime(history.index).tz_localize(None)
missing = [t for t in tickers + [BENCHMARK] if t not in history or history[t].isna().all()]
assert not missing, f"No price data for {missing}. Check the ticker symbols."

# Fill any blank avg_cost with the closing price on the buy date (or the next trading day)
def close_on(ticker, day):
    i = min(history.index.searchsorted(day), len(history) - 1)
    return float(history[ticker].iloc[i])

holdings["avg_cost"] = [c if pd.notna(c) else close_on(t, d)
                        for t, d, c in zip(holdings["ticker"], holdings["buy_date"], holdings["avg_cost"])]
holdings["avg_cost"] = holdings["avg_cost"].astype(float)

holdings = revalue(holdings, get_prices(tickers))
print(summary(holdings))
display(style_table(holdings))

## 2. Snapshot: average cost vs. current price

In [ ]:
def price_comparison_chart(df, title="Average cost vs. current price"):
    df = df.sort_values("return_pct", ascending=False)
    t = df["ticker"]
    colors = np.where(df["price"] >= df["avg_cost"], "#2E9E5B", "#D64545")

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.65, 0.35],
                        vertical_spacing=0.08,
                        subplot_titles=("Price per share (log scale)", "Gain / loss vs. your average cost"))
    fig.add_trace(go.Bar(x=t, y=df["avg_cost"], name="Your average cost", marker_color="#9AA5B1",
                         text=[f"${v:,.2f}" for v in df["avg_cost"]], textposition="outside"), row=1, col=1)
    fig.add_trace(go.Bar(x=t, y=df["price"], name="Current price", marker_color=colors,
                         text=[f"${v:,.2f}" for v in df["price"]], textposition="outside"), row=1, col=1)
    fig.add_trace(go.Bar(x=t, y=df["return_pct"], marker_color=colors, showlegend=False,
                         text=[f"{v:+.1%}" for v in df["return_pct"]], textposition="outside",
                         customdata=[f"{money(g)} on {s:,.4g} shares" for g, s in zip(df["gain"], df["shares"])],
                         hovertemplate="%{x}<br>%{y:+.1%}<br>%{customdata}<extra></extra>"),
                  row=2, col=1)
    lim = max(df["return_pct"].abs().max(), 0.01) * 1.4          # room so labels aren't cut off
    fig.update_layout(title=title, barmode="group", height=650, hovermode="x unified",
                      legend=dict(orientation="h", y=1.08, x=0))
    fig.update_yaxes(type="log", tickprefix="$", row=1, col=1)   # log so $3 and $750 stocks both show
    fig.update_yaxes(tickformat=".0%", range=[-lim, lim], zeroline=True, zerolinecolor="#666", row=2, col=1)
    return fig

price_comparison_chart(holdings).show()

## 3. Performance analytics

- **Time-weighted return** removes the effect of when money was added, so it measures stock selection. This is the figure to compare against the benchmark.
- **XIRR (money-weighted return)** includes the timing of each purchase, so it is the investor's personal annualized return.
- Returns are price-only (dividends not included).

In [ ]:
from scipy.optimize import brentq

dates = history.index[history.index >= holdings["buy_date"].min()]
buy_idx = np.minimum(dates.searchsorted(holdings["buy_date"]), len(dates) - 1)   # weekend buys move to next trading day
buy_day = dates[buy_idx]

# Shares held each day, portfolio value, and money invested each day
shares_daily = pd.DataFrame({t: np.where(dates >= d, s, 0.0)
                             for t, d, s in zip(holdings["ticker"], buy_day, holdings["shares"])}, index=dates)
value = (shares_daily * history[holdings["ticker"]].reindex(dates)).sum(axis=1)
flows = (pd.Series(holdings["cost_basis"].values, index=buy_day).groupby(level=0).sum()
         .reindex(dates, fill_value=0))
invested = flows.cumsum()

# Daily time-weighted returns for the portfolio and the benchmark
prev = value.shift(1)
port_ret = ((value - flows) / prev - 1).where(prev > 0).dropna()
bench_ret = history[BENCHMARK].reindex(dates).pct_change().reindex(port_ret.index)
growth, bench_growth = (1 + port_ret).cumprod() - 1, (1 + bench_ret).cumprod() - 1

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08,
                    subplot_titles=("Portfolio value vs. money invested", f"Cumulative return vs. {BENCHMARK}"))
fig.add_trace(go.Scatter(x=dates, y=value, name="Portfolio value", line=dict(color="#2E75B6", width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=dates, y=invested, name="Money invested", line=dict(color="#9AA5B1", dash="dot"),
                         line_shape="hv"), row=1, col=1)
fig.add_trace(go.Scatter(x=growth.index, y=growth, name="Portfolio (time-weighted)",
                         line=dict(color="#16803C", width=2)), row=1, col=2)
fig.add_trace(go.Scatter(x=bench_growth.index, y=bench_growth, name=BENCHMARK,
                         line=dict(color="#B42318", width=1.5)), row=1, col=2)
fig.update_yaxes(tickprefix="$", row=1, col=1)
fig.update_yaxes(tickformat=".0%", row=1, col=2)
fig.update_layout(height=450, hovermode="x unified", legend=dict(orientation="h", y=-0.15))
fig.show()


def metrics(r, bench):
    g = (1 + r).cumprod()
    ann_ret = g.iloc[-1] ** (252 / len(r)) - 1
    ann_vol = r.std() * np.sqrt(252)
    return {"Total return": g.iloc[-1] - 1,
            "Annualized return": ann_ret,
            "Annualized volatility": ann_vol,
            "Sharpe ratio": (ann_ret - RISK_FREE_RATE) / ann_vol,
            "Max drawdown": (g / g.cummax() - 1).min(),
            "Beta": r.cov(bench) / bench.var(),
            "Correlation": r.corr(bench)}


m = pd.DataFrame({"Portfolio": metrics(port_ret, bench_ret), BENCHMARK: metrics(bench_ret, bench_ret)})
pct_rows = ["Total return", "Annualized return", "Annualized volatility", "Max drawdown"]
display(m.style.format("{:.2%}", subset=pd.IndexSlice[pct_rows, :])
               .format("{:.2f}", subset=pd.IndexSlice[["Sharpe ratio", "Beta", "Correlation"], :]))


def xirr(cashflows):
    d0 = cashflows[0][0]
    npv = lambda rate: sum(a / (1 + rate) ** ((d - d0).days / 365) for d, a in cashflows)
    return brentq(npv, -0.9999, 100)

cfs = sorted([(d, -c) for d, c in zip(holdings["buy_date"], holdings["cost_basis"])]
             + [(pd.Timestamp.now().normalize(), holdings["value"].sum())])
try:
    print(f"Money-weighted annual return (XIRR): {xirr(cfs):.2%}")
except ValueError:
    print("XIRR could not be computed for these cash flows.")

## 4. Live mode

Refreshes prices, both charts and the holdings table every `REFRESH_SECONDS` while the market is open, and every `CLOSED_REFRESH_SECONDS` when it is closed. Times are shown in Los Angeles time. Market holidays are not detected.

In [ ]:
REFRESH_SECONDS = 60             # update interval while the market is open
CLOSED_REFRESH_SECONDS = 900     # update interval while it's closed (15 min)
RUN_MINUTES = 60                 # how long to keep updating
INCLUDE_EXTENDED_HOURS = False   # True to track pre-market and after-hours prices


# ---------- Chart 1: % change and dollar gain vs. your average cost ----------
def cost_vs_price_chart(df, title="Average cost vs. current price"):
    df = df.sort_values("return_pct")              # best performer ends up on top
    colors = np.where(df["return_pct"] >= 0, "#2E9E5B", "#D64545")
    pct_labels = [f"{r:+.2%}   ${c:,.2f} → {p:,.2f}"     # one "$" per label (two would render as math)
                  for r, c, p in zip(df["return_pct"], df["avg_cost"], df["price"])]

    fig = make_subplots(rows=1, cols=2, shared_yaxes=True, column_widths=[0.68, 0.32],
                        horizontal_spacing=0.03,
                        subplot_titles=("% change from your average cost   (avg cost → current price)",
                                        "Dollar gain / loss"))
    fig.add_trace(go.Bar(y=df["ticker"], x=df["return_pct"], orientation="h", marker_color=colors,
                         text=pct_labels, textposition="outside", cliponaxis=False,
                         customdata=[f"<b>{t}</b><br>Avg cost ${c:,.2f} → now {p:,.2f}<br>Change {r:+.2%}"
                                     for t, c, p, r in zip(df["ticker"], df["avg_cost"], df["price"], df["return_pct"])],
                         hovertemplate="%{customdata}<extra></extra>"),
                  row=1, col=1)
    fig.add_trace(go.Bar(y=df["ticker"], x=df["gain"], orientation="h", marker_color=colors,
                         text=[money(g) for g in df["gain"]], textposition="outside", cliponaxis=False,
                         hovertemplate="<b>%{y}</b><br>%{x:+$,.2f}<extra></extra>"),
                  row=1, col=2)

    pct_lim = max(df["return_pct"].abs().max(), 0.01) * 2.6   # padding so labels never get cut off
    usd_lim = max(df["gain"].abs().max(), 0.5) * 1.9
    fig.update_xaxes(range=[-pct_lim, pct_lim], tickformat=".1%", zeroline=True,
                     zerolinecolor="#444", zerolinewidth=2, row=1, col=1)
    fig.update_xaxes(range=[-usd_lim, usd_lim], tickformat="$,.2f", zeroline=True,
                     zerolinecolor="#444", zerolinewidth=2, row=1, col=2)
    fig.update_layout(title=title, showlegend=False, height=140 + 55 * len(df),
                      bargap=0.35, margin=dict(l=70, r=30, t=90, b=40),
                      uniformtext=dict(minsize=11, mode="show"))
    return fig


# ---------- Chart 2: money spent (inner pie) with profit / loss ring ----------
def blend(c1, c2, t):
    """Mix two hex colors, t=0 gives c1 and t=1 gives c2."""
    a = np.array([int(c1[i:i + 2], 16) for i in (1, 3, 5)])
    b = np.array([int(c2[i:i + 2], 16) for i in (1, 3, 5)])
    r, g, bl = (a + (b - a) * t).round().astype(int)
    return f"#{r:02x}{g:02x}{bl:02x}"


def spend_pnl_pie(df, title="Money spent per stock, with profit / loss ring"):
    df = df.sort_values("cost_basis", ascending=False).reset_index(drop=True)
    total_spent, total_gain = df["cost_basis"].sum(), df["gain"].sum()

    max_move = max(df["return_pct"].abs().max(), 0.01)       # deeper color = bigger % move
    ring_colors = [blend("#C6EBCF", "#16803C", min(abs(r) / max_move, 1)) if r >= 0
                   else blend("#F9C9C4", "#B42318", min(abs(r) / max_move, 1))
                   for r in df["return_pct"]]
    inner_colors = ["#1F4E79", "#2E75B6", "#4A90C8", "#6BAED6", "#8FC1E3",
                    "#5B7FA6", "#7895B2", "#A3BBD3", "#3D6A99", "#B7CCE0"]
    inner_colors = (inner_colors * (len(df) // len(inner_colors) + 1))[:len(df)]
    common = dict(labels=df["ticker"], values=df["cost_basis"], sort=False,
                  direction="clockwise", rotation=90)

    fig = go.Figure()
    fig.add_trace(go.Pie(                                    # inner circle: money spent
        **common, name="Spent", domain=dict(x=[0, 1], y=[0.13, 0.87]),
        marker=dict(colors=inner_colors, line=dict(color="white", width=2)),
        text=[f"{t}<br>${c:,.2f}<br>{c / total_spent:.0%}" if c / total_spent >= 0.05 else ""
              for t, c in zip(df["ticker"], df["cost_basis"])],
        textinfo="text", textposition="inside", insidetextorientation="horizontal",
        hovertemplate="<b>%{label}</b><br>Spent $%{value:,.2f}<br>%{percent} of total<extra></extra>",
    ))
    fig.add_trace(go.Pie(                                    # outer ring: profit / loss
        **common, name="P/L", hole=0.80, domain=dict(x=[0, 1], y=[0, 1]),
        marker=dict(colors=ring_colors, line=dict(color="white", width=2)),
        text=[f"{t} {money(g)} ({r:+.1%})" for t, g, r in zip(df["ticker"], df["gain"], df["return_pct"])],
        textinfo="text", textposition="outside",
        customdata=[f"<b>{t}</b><br>P/L {money(g)} ({r:+.2%})"
                    for t, g, r in zip(df["ticker"], df["gain"], df["return_pct"])],
        hovertemplate="%{customdata}<extra></extra>",
    ))
    fig.update_layout(                                       # one "$" per text line (two render as math)
        title=dict(text=title, x=0, xref="paper", y=0.97),
        annotations=[
            dict(text=f"Spent ${total_spent:,.2f}", x=0, y=1.10, xref="paper", yref="paper",
                 xanchor="left", showarrow=False, font=dict(size=14)),
            dict(text=f"Total P/L {money(total_gain)} ({total_gain / total_spent:+.2%})",
                 x=0, y=1.06, xref="paper", yref="paper", xanchor="left", showarrow=False,
                 font=dict(size=14, color="#16803C" if total_gain >= 0 else "#B42318")),
        ],
        showlegend=False, height=760, margin=dict(t=140, b=60, l=40, r=40))
    return fig


# ---------- Market hours (US exchanges run on New York time) ----------
def market_status(now):
    ny = now.tz_convert("America/New_York")
    if ny.weekday() >= 5:
        return "Closed (weekend)", False
    m = ny.hour * 60 + ny.minute
    if 9 * 60 + 30 <= m < 16 * 60:
        return "Open", True
    if 4 * 60 <= m < 9 * 60 + 30:
        return "Pre-market", INCLUDE_EXTENDED_HOURS
    if 16 * 60 <= m < 20 * 60:
        return "After hours", INCLUDE_EXTENDED_HOURS
    return "Closed", False


# ---------- Live loop ----------
stop_at = time.time() + RUN_MINUTES * 60
try:
    while time.time() < stop_at:
        now = pd.Timestamp.now(tz=TZ)
        status, active = market_status(now)
        wait = REFRESH_SECONDS if active else CLOSED_REFRESH_SECONDS

        live = get_prices(tickers, intraday=True, extended=INCLUDE_EXTENDED_HOURS)
        snap = revalue(holdings, live)
        stamp = f"{now:%I:%M:%S %p %Z}"

        clear_output(wait=True)
        next_txt = f"{wait // 60} min" if wait >= 60 else f"{wait}s"
        print(f"Updated {now:%a %b %d} {stamp}   |   Market: {status}   |   next update in {next_txt}")
        print(summary(snap))
        missing = live.index[live.isna()].tolist()
        if missing:
            print(f"No live price for {missing}, showing last known price")

        cost_vs_price_chart(snap, f"Live: average cost vs. current price ({stamp})").show()
        spend_pnl_pie(snap, f"Live: money spent per stock, with profit / loss ring ({stamp})").show()
        display(style_table(snap))

        time.sleep(max(0, min(wait, stop_at - time.time())))
    print("Live mode finished. Run the cell again to restart.")
except KeyboardInterrupt:
    print("Live mode stopped.")